<a href="https://colab.research.google.com/github/FadouaBOUAFIF/Cours_Traitement_images_TP/blob/main/TP2_Weber_Quantification_FFT2D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP N°2 : Loi de Weber, Quantification et Transformée de Fourier 2D
**Objectifs:**
* Comprendre la différence entre les types de données et la dynamique d'une image sous Python.
* Étudier l'effet de la **quantification** (réduction du nombre de niveaux de gris) et l'apparition des **faux contours**.
* Vérifier expérimentalement la **loi de Weber** (perception humaine des contrastes).
* Manipuler la **Transformée de Fourier Discrète 2D (FFT2)** et comprendre le rôle fondamental du **module** et de la **phase**.
* Implémenter et comparer des opérateurs de traitement d'image dans le **domaine spatial** et dans le **domaine fréquentiel**.

---

### 🛠️ 1. Introduction et préparation de l'environnement


Exécutez la cellule ci-dessous pour charger les bibliothèques et les images standards.

In [1]:
import numpy as np    # traitement matriciel
import matplotlib.pyplot as plt    #visualisation des images
from skimage import data, color, transform  #traitement d'image: données, coleur, transformation géométrique

# Chargement des images de référence en niveaux de gris (uint8 [0..255])
cameraman = data.camera()  # Image 512x512 uint8

# Chargement d'une seconde image de référence (Astronaute convertie en niveaux de gris, redimensionnée en 512x512)
img2_float = color.rgb2gray(data.astronaut()) # float64 [0..1]
img2 = (transform.resize(img2_float, (512, 512)) * 255)
print(f"Cameraman : shape={cameraman.shape}")
print(f"Image 2   : shape={img2.shape}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(cameraman, cmap='gray')
axes[0].set_title("Cameraman (512x512)")
axes[0].axis('off')

axes[1].imshow(img2, cmap='gray')
axes[1].set_title("Image 2 - Portrait (512x512)")
axes[1].axis('off')
plt.show()



---
## Partie 2 : Quantification

Une image codée sur $8$ bits possède $2^8 = 256$ niveaux de gris. Quantifier l'image sur $N$ niveaux revient à réduire le pas de quantification $q = \frac{256}{N}$.

**Travail demandé :**
1. Complétez la fonction `quantifier(image, niveaux)` pour quantifier l'image `cameraman` sur **128, 64, 32, 16, 8, 4 et 2 niveaux de gris**.
2. Affichez les images résultantes et déterminez le seuil minimal de niveaux de gris à partir duquel des **faux contours** (effet d'escalier dans les zones dégradées comme le ciel) apparaissent.
3. Commentez l'influence de la variation du pas de quantification.

In [ ]:
def quantifier(image, niveaux):
    """
    Quantifie une image uint8 [0..255] sur 'niveaux' niveaux de gris.
    Indice : calculez le pas q = 256 // niveaux, effectuez une division entière (image // q) * q
    """
    # À compléter :
    q = 256 // niveaux
    img_q = (image // q) * q
    return img_q

niveaux_liste = [128, 64, 32, 16, 8, 4, 2]

# À compléter : Affichez l'image originale et les 7 images quantifiées dans une grille (2 lignes x 4 colonnes)
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.ravel()

axes[0].imshow(cameraman, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Originale (256 niveaux)")
axes[0].axis('off')

for idx, n in enumerate(niveaux_liste):
    # Appelez quantifier(cameraman, n) et affichez dans axes[idx + 1]
    pass

plt.tight_layout()
plt.show()

**✍️ Vos réponses et commentaires (Partie 2 - Quantification) :**
> *Double-cliquez ici pour indiquer le seuil d'apparition des faux contours et commenter l'influence du pas de quantification...*

---
## Partie 3 : Loi de Weber

La **loi de Weber** stipule que le seuil différentiel de luminance $\Delta L = L_2 - L_1$ perceptible par l'œil humain est proportionnel à la luminance de référence :
$$\frac{\Delta L}{L_1} = k \quad (\text{constante de Weber})$$

**Géométrie de la mire à générer :**
* Image carrée de taille **$600 \times 600$ pixels** avec un fond uniforme d'intensité **$L_b$**.
* Au centre de l'image, un carré de taille **$160 \times 160$ pixels** (donc de la ligne `220` à `380`, et de la colonne `220` à `380`).
* Ce carré central est divisé verticalement en deux rectangles égaux de **$160 \times 80$ pixels** :
  * Moitié gauche (colonnes `220` à `300`) : intensité **$L_1$**.
  * Moitié droite (colonnes `300` à `380`) : intensité **$L_2$**.

**Travail demandé :**
1. Écrivez la fonction `weber(L1, L2, Lb)` renvoyant cette matrice 2D.
2. Prenez **$L_b = 10$** et **$L_1 = 50$** (par exemple). En faisant varier progressivement $L_2 > L_1$, déterminez la différence minimale $\Delta L = L_2 - L_1$ permettant de distinguer la séparation verticale entre les deux moitiés, puis calculez la constante $k = \frac{\Delta L}{L_1}$.
3. Reprenez la même démarche avec un fond clair **$L_b = 200$** (et par exemple $L_1 = 150$). Commentez.

In [ ]:
def weber(L1, L2, Lb):
    """
    Génère une matrice 600x600 avec un fond Lb et un carré central 160x160
    divisé en deux moitiés verticales d'intensités L1 et L2.
    """
    # À compléter :
    img = np.full((600, 600), Lb)
    # Remplir la moitié gauche (L1) et la moitié droite (L2) du carré central [220:380, 220:380]

    return img

# Testez vos valeurs de L1, L2, Lb ici :
Lb = 10
L1 = 50
L2 = 55  # À ajuster expérimentalement

mire = weber(L1, L2, Lb)

plt.figure(figsize=(5, 5))
plt.imshow(mire, cmap='gray', vmin=0, vmax=255)
plt.title(f"Mire de Weber (Lb={Lb}, L1={L1}, L2={L2})")
plt.axis('off')
plt.show()

**✍️ Vos réponses et commentaires (Partie 3 - Loi de Weber) :**
> *Double-cliquez ici pour noter vos valeurs expérimentales de $\Delta L$ et $k = \Delta L / L_1$ pour $L_b = 10$ et $L_b = 200$, et commenter l'influence de la luminance du fond $L_b$...*

---
## Partie 4 : Transformée de Fourier Discrète Bidimensionnelle (FFT 2D)

Sous Python, le module `numpy.fft` fournit les fonctions `np.fft.fft2()`, `np.fft.ifft2()` et `np.fft.fftshift()`.
En coordonnées polaires, chaque coefficient complexe du spectre s'écrit :
$$F(u, v) = |F(u, v)| \cdot e^{j \varphi(u, v)}$$
où $|F(u, v)| = \text{np.abs}(F)$ est le **module** (amplitude) et $\varphi(u, v) = \text{np.angle}(F)$ est la **phase**.

**Travail demandé :**
1. Calculez la FFT 2D de la mire de Weber (ou de `cameraman`), puis appliquez la FFT inverse (`ifft2`) et affichez le résultat (pensez à prendre la partie réelle `np.real()`).
2. Calculez les FFT de `cameraman` et `img2`. Affichez leur spectre d'amplitude centré (`fftshift`) en échelle logarithmique : $\log(1 + |F_{shift}|)$.
3. Reconstruisez `cameraman` en **annulant la phase** ($\varphi(u,v) = 0$, soit $F_{mod} = |F|$) et appliquez `ifft2`.
4. Reconstruisez `cameraman` en **fixant le module constant** ($|F(u,v)| = 1$, soit $F_{phase} = e^{j \varphi(u,v)}$) et appliquez `ifft2`.
5. Créez une image hybride en combinant le **module de `img2`** et la **phase de `cameraman`** : $F_{hybride} = |F_{img2}| \cdot e^{j \varphi_{cameraman}}$.
6. Concluez sur les rôles respectifs du module et de la phase dans une image.

In [ ]:
# 1 & 2. Calcul des FFT 2D de cameraman et img2
F_cam = np.fft.fft2(cameraman)
F_img2 = np.fft.fft2(img2)

# Affichage des spectres centrés en échelle logarithmique
spectre_cam = np.log(1 + np.abs(np.fft.fftshift(F_cam)))
spectre_img2 = np.log(1 + np.abs(np.fft.fftshift(F_img2)))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(spectre_cam, cmap='gray')
axes[0].set_title("Spectre centré - Cameraman")
axes[1].imshow(spectre_img2, cmap='gray')
axes[1].set_title("Spectre centré - Image 2")
plt.show()

# À compléter :
# 3. Annulation de la phase de cameraman (garder uniquement np.abs(F_cam))

# 4. Module constant égal à 1 pour cameraman (garder uniquement np.exp(1j * np.angle(F_cam)))

# 5. Image hybride : Module de img2 + Phase de cameraman


**✍️ Vos conclusions (Partie 4 - Rôle du module et de la phase) :**
> *Double-cliquez ici pour rédiger votre analyse sur l'importance de la phase par rapport au module...*

---
## Partie 5 : Manipulation dans le domaine fréquentiel vs spatial

Pour chacune des 4 opérations suivantes, implémentez **deux versions** (l'une travaillant dans le **domaine spatial** $I(x,y)$, l'autre dans le **domaine fréquentiel** $F(u,v)$) et appliquez-les sur `img2` :

1. **`moyenne(I)`** : renvoie la valeur moyenne de l'image $I$.
   * *Théorie :* $F(0, 0) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} I(x, y) = M \cdot N \cdot \bar{I}$.
2. **`symcentre(I)`** : retourne la symétrie centrale de $I$ par rapport à son centre.
   * *Théorie :* La symétrie spatiale $I(-x, -y)$ correspond au conjugué complexe $F^*(u, v)$ (ou à une double FFT : $\mathcal{F}\{\mathcal{F}\{I\}\}$ à un facteur d'échelle près).
3. **`contraste(I, alpha)`** : multiplie par $\alpha$ les écarts à la moyenne (fréquences non-nulles) tout en conservant la composante continue $F(0, 0)$.
   * *Théorie :* $I_{out}(x,y) = \bar{I} + \alpha (I(x,y) - \bar{I}) \iff F_{out}(u,v) = \alpha F(u,v)$ pour $(u,v) \neq (0,0)$ et $F_{out}(0,0) = F(0,0)$.
4. **`translation(I, dx, dy)`** : opère une translation circulaire de $I$ par un vecteur $(dx, dy)$.
   * *Théorie :* Théorème du retard : $I(x - dx, y - dy) \iff F(u, v) \cdot e^{-j 2\pi \left(\frac{u \cdot dx}{M} + \frac{v \cdot dy}{N}\right)}$.

In [ ]:
# 1. MOYENNE
def moyenne_spatiale(I):
    return np.mean(I)

def moyenne_frequentielle(I):
    # À compléter à partir de F[0, 0]
    pass

# 2. SYMÉTRIE CENTRALE
def symcentre_spatiale(I):
    return I[::-1, ::-1]

def symcentre_frequentielle(I):
    # À compléter : utilisez np.conj(np.fft.fft2(I)) puis ifft2
    pass

# 3. CONTRASTE
def contraste_spatial(I, alpha):
    # À compléter : moy + alpha * (I - moy)
    pass

def contraste_frequentiel(I, alpha):
    # À compléter : multipliez F par alpha sauf F[0, 0]
    pass

# 4. TRANSLATION CIRCULAIRE
def translation_spatiale(I, dx, dy):
    return np.roll(I, shift=(dy, dx), axis=(0, 1))

def translation_frequentielle(I, dx, dy):
    # À compléter : multipliez F par l'exponentielle complexe de déphasage
    pass

# Zone de test et d'affichage comparatif :


**✍️ Conclusion et synthèse des résultats obtenus:**
> *Double-cliquez ici pour rédiger un synthèse des résultats obtenus...*